<a href="https://colab.research.google.com/github/Iberasa3/Hollow-Hornet/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import pandas as pd

# El separador '\t' le dice a pandas que use el tabulador
df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit

# Ahora sí, vamos a comprobar que se han separado
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()[:5]}...")

Número de columnas: 50
Nombres de las columnas: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


Efectivamente este es mi proyecto de avispas, de momento no tengo mucho más que añadir. Cuando termine de limpiar este dataset proyectaré las coordenadas en GEE

Un par de apuntes para mí mismo, probablemente acabemos considerando a todas las subespecies de avispa asiatica como una sola para no tener que liarnos entre las distintas subespecies.

Varios factores que tengo que tener en cuenta para poder filtrar bien los datos que se me están presentando:

1- Los datos anteriores a 2005, si los hay, no hay que tenerlos en cuenta porque Vespa velutina entró a europa en 2004 (¿hecho?)

2- Cuidado con null island (hecho)

3- cuidado con los rangos en los que el avistamiento tiene gran índice de error (hecho)

4- cuidado con los avistamientos no-salvajes (hecho, preservados eliminados)

5- hay que chekiar las diferentes subespecies por si acaso (hecho)

6- hay que chekiar solo los que sean de europa (hecho)


Tengo que leer algunos papers para que esto parezca más académico, pero tampoco pasarme 150 días leyendo papers para justificar toda la literatura


In [49]:
df['infraspecificEpithet'].unique() #Si la celda está vacía entonces asumimos que es la versión común de la vespa

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [50]:
paises_europa = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL'] #El análisis se basa en europa así que nos quedamos solo con los países europeos. La mayoría de registros de todas formas son europeos.
df = df[df['countryCode'].isin(paises_europa)]
print(f"Pasamos de {len(df)} a {len(df)} registros.")

Pasamos de 372895 a 372895 registros.


In [51]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor
conteo_paises = df['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    201699
NL     72038
FR     54466
CH     20092
DE     17921
ES      3518
PT      2251
LU       642
IT       211
IE        37
GB        20
Name: count, dtype: int64


In [52]:
#Quitamos todas las coordenadas basura y los nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Registros originales: {len(df)}")
print(f"Registros después de haber quitado los nulls y los null-islands: {len(df)}")

Registros originales: 372450
Registros después de haber quitado los nulls y los null-islands: 372450


In [53]:
#Quitamos las instancias en las que la incertidumbre es superior a 1km. Habría que ver por qué coño existen incertidumbres tan grandes
umbral_maximo = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]

Hay que tener cuidado con las instancias que se hayan tomado en el mismo instante, puede ser que un pibe haya apuntado varias veces a una misma colmena o incluso a una misma avispa, y que por ello haya zonas sobrerepresentadas.
Ahora habría que ver cómo chota hacemos esto del spatial thinning.
Como hemos descargado la versión simple del Darwin Core igual tenemos datos que nos ayudan con esto. De hecho igual en un futuro lo hago directamente con DWC

In [54]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

['HUMAN_OBSERVATION']


Vamos con el spatial thinning.

Podemos aplicar una agrupación por celda, útil para evitar sobremuestreos pero falla para detectar zonas más idóneas en las que las avispas hayan podido construir sus colmenas. Luego podemos sumarlo a un check de correlación para ver si nos ha salido bien.

La cosa es que si vamos a aplicar un random forest se asume que cada dato va a ser independiente, el modelo no va a saber que los distintos puntos que están cerca entre sí pueden reflejar una colmena próxima. Además al reflejar los puntos en el propio mapa la resolución no va a dar para más, así que creo que hacer el spatial thinning sin tener en cuenta la posibilidad de colmena es, de momento, suficiente. Esto es un modelo de distribución, no uno de abundancia.

¿Cuánto vuela una avispa desde su nido?
Por lo que he leído en internet de fuentes fiables, todas coinciden en que rara vez sobrepasan 1km o el kilometro y medio desde sus nidos, así que si hay algún avistamiento de avispa debe de haber un nido relativamente cerca casi seguro



In [44]:
num_duplicados = df.duplicated().sum()
print(f"Registros con el mismo gbifID: {num_duplicados}")
df.drop_duplicates(subset='gbifID', inplace=True)

Registros con el mismo gbifID: 0


Hemos comprobado también que no haya avistamientos con un ID repetido. Se asume que los estándares de Darwin Core no permiten duplicados en sus datasets, pero por si acaso

In [45]:
# 0.015 grados de latitud son aproximadamente 1.1 km, así que cada celda va a tener aprox 1,1km x 1,1km
# Es una resolución estándar para modelos climáticos de 1km.
res = 0.015

# Dividimos por la resolución, redondeamos al entero más cercano y volvemos a multiplicar.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset original: {len(df)} registros")
print(f"Dataset tras thinning (1km): {len(df)} registros")
print(f"Registros eliminados: {len(df) - len(df)}")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

Dataset original: 29307 registros
Dataset tras thinning (1km): 29307 registros
Registros eliminados: 0


,decimalLatitude,lat_grid,decimalLongitude,lon_grid
2,51.11106,51.105,4.07784,4.080
3,51.21826,51.225,2.90049,2.895
4,50.86031,50.865,4.62720,4.620
5,51.03947,51.045,3.74158,3.735
6,50.95183,50.955,3.68754,3.690


In [46]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor, estos son las casillas donde ha habido avistamiento
conteo_paises = df['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    10519
FR     8792
DE     7482
ES     1247
PT      760
LU      184
NL      124
CH       94
IT       91
IE        9
GB        5
Name: count, dtype: int64


In [47]:
import sys
import geemap

columnas_necesarias = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[columnas_necesarias].copy()

df = df.dropna(subset=['decimalLatitude', 'decimalLongitude']) #Eliminamos los valores na

puntos_geospatiales = geemap.pandas_to_ee(
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

Map = geemap.Map()
Map.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(puntos_geospatiales, 6)
Map

Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

Ahora toca hacer que un modelo randomForest me aprenda las características de los sitios donde puede haber más avispas Vespa Velutina.

Hasta aquí hemos:
1- Conseguido el dataset desde GBIF, que registra observaciones de la Vespa velutina en un formato Darwin core (aunque de momento con la versión procesada nos sirve)

2- Hemos filtrado el dataset (1) quedándonos solo con los países de europa (las condiciones de Asía no nos sirven si son parecidas?¿?), (2) filtrando y eliminando las coordenadas no-útiles, (3) eliminando las instancias cuya tasa de error es superior a 1km, (4) quedándonos solo con las especies vistas en estado silvestre, (5) quedándonos solo con los avistamientos realizados a partir de 2005, y (6), eliminando las posibles observaciones duplicadas.

3- Hemos hecho una refracción espacial de 1,6km para que ciertos avistamientos no se vean representados, convirtiendo el mapa en casillas cuyos valores son o 1 (en cuyo caso la avispa está presente) o 0, en cuyo caso no lo está

4- Hemos reflejado dichas casillas en el mapa con GEE.

El siguiente paso es aprender los valores geográficos de las casillas en las que SÍ hay avispa y de las casillas en las que NO las hay. Luego aplicaremos un modelo random forest para poder predecir futuras expansiones. La casilla 0 estará representada por pseudoausencias, que enseñarán al modelo en qué condiciones NO hay avispas.

PROBLEMA: ¿CÓMO SABEMOS SI NO HAY AVISPAS PORQUE TODAVÍA NO HAN LLEGADO O PORQUE LAS CONDICIONES NO SON FAVORABLES?

Plus sería buena idea descargarme el dataset limpio para no tener que andar limpiándolo cada vez